In [ ]:
!pip install bertopic pythainlp pandas
!pip install -U sentence-transformers

In [2]:
import pandas as pd

df = pd.read_csv("services/visualization/assets/my_thai_topics_results_bert_cv_23.csv")
df.head()

,id,platform,platform_type,url,content_type,timestamp,scraper_module,text,scrape_date,source_file,tokens,clean,topic,topic_name
0,UgxTzXyCxlnww6zFwCR4AaABAg,youtube,video,https://www.youtube.com/watch?v=4zeNLkX0TPg,comment,2022-11-12 00:00:00,yt-dlp,หากต้องการปรึกษาแบบส่วนตัว หรือ สนใจคอร์ส สามา...,2025-11-12 03:48:57,/Users/nanphattongsirisukool/gender-bias-detec...,"['หาก', 'ต้อง', 'การปรึกษา', 'แบบ', 'ส่วนตัว',...","['หาก', 'ต้อง', 'การปรึกษา', 'แบบ', 'ส่วนตัว',...",23,23_10 17 5000_10 2000_10 2000 3000_15 facebook...
1,UgxyZgks2H3UdlT9CGN4AaABAg,youtube,video,https://www.youtube.com/watch?v=4zeNLkX0TPg,comment,2025-11-09 00:00:00,yt-dlp,ขอบคุณมากค่ะ,2025-11-12 03:48:57,/Users/nanphattongsirisukool/gender-bias-detec...,"['ขอบคุณ', 'มาก', 'ค่ะ']","['ขอบคุณ', 'มาก', 'ค่ะ']",86,86_samakkarn 2_fc chatgpt attitude_samakkarn 2...
2,UgzmW3OaNb3BLVSnMFd4AaABAg,youtube,video,https://www.youtube.com/watch?v=4zeNLkX0TPg,comment,2025-11-08 00:00:00,yt-dlp,ขอบคุณค่ะ,2025-11-12 03:48:57,/Users/nanphattongsirisukool/gender-bias-detec...,"['ขอบคุณ', 'ค่ะ']","['ขอบคุณ', 'ค่ะ']",115,115_thank you gamego_phoenix twis arena_twis a...
3,Ugy-XbHKm2hV5sx9fop4AaABAg,youtube,video,https://www.youtube.com/watch?v=4zeNLkX0TPg,comment,2025-07-12 00:00:00,yt-dlp,อยู่คนเดียวสบายที่สุด,2025-11-12 03:48:57,/Users/nanphattongsirisukool/gender-bias-detec...,"['อยู่', 'คนเดียว', 'สบาย', 'ที่สุด']","['อยู่', 'คนเดียว', 'สบาย', 'ที่สุด']",383,383____
4,UgwKfQUzYLxmF6bYHxl4AaABAg,youtube,video,https://www.youtube.com/watch?v=4zeNLkX0TPg,comment,2025-07-12 00:00:00,yt-dlp,กำลังเจอค่ะ,2025-11-12 03:48:57,/Users/nanphattongsirisukool/gender-bias-detec...,"['กำลัง', 'เจอ', 'ค่ะ']","['กำลัง', 'เจอ', 'ค่ะ']",414,414____


In [3]:
import ast
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

# 1. Setup Data
# 'raw_text': The full natural Thai sentences (Best for the AI model)
# 'tokens': Your pre-tokenized lists (Best for the Keyword count)
raw_text_list = df["text"].tolist()
df["tokens"] = df["tokens"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
token_text_list = df["tokens"].apply(lambda x: " ".join(x) if isinstance(x, list) else x).tolist()

/Users/nanphattongsirisukool/anaconda3/envs/gbd/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

# 2. Load the "Modern" Embedding Model
# BAAI/bge-m3 is currently SOTA for multilingual retrieval/similarity
print("Loading model and encoding embeddings...")
embedding_model = SentenceTransformer('BAAI/bge-m3')

# Generate embeddings from the RAW TEXT (this captures the semantic meaning best)
embeddings = embedding_model.encode(raw_text_list, show_progress_bar=True)

Loading model and encoding embeddings...


OSError: Can't load the model for 'BAAI/bge-m3'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'BAAI/bge-m3' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

In [ ]:

# 3. Create the "Trust My Tokens" Vectorizer
# This will be applied to the 'token_text_list' we pass later
vectorizer_model = CountVectorizer(
    tokenizer=lambda x: x.split(),
    token_pattern=None,
    ngram_range=(2, 3) # (1,2) is great for Thai to capture compound words
)


In [ ]:

# 4. Initialize BERTopic
# We don't pass an embedding_model here because we will pass pre-computed embeddings
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    verbose=True
)

# 5. The Magic Step: Mix them!
# Pass the TOKEN list as the docs (for counting words)
# Pass the RAW TEXT EMBEDDINGS for clustering
topics, probs = topic_model.fit_transform(token_text_list, embeddings=embeddings)


2025-12-02 11:12:10,606 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-02 11:12:46,033 - BERTopic - Dimensionality - Completed ✓
2025-12-02 11:12:46,035 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-02 11:12:48,999 - BERTopic - Cluster - Completed ✓
2025-12-02 11:12:49,011 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-02 11:12:49,537 - BERTopic - Representation - Completed ✓


In [ ]:

# 6. View Results
print(topic_model.get_topic_info().head())

   Topic  Count                                               Name  \
0     -1  14732                             -1_2 d_pick me_2 3_2 2   
1      0    342     0_black th_pink black th_pink black_roly emoji   
2      1    298                   1_exo exo_mv exo_exo mv_kris exo   
3      2    295  2_feminine feminine_bruhhh am not_am not_am no...   
4      3    215  3_1 5_thailand s most_the world yeahhhhh_s mos...   

                                      Representation  \
0  [2 d, pick me, 2 3, 2 2, noowa ammy, gold digg...   
1  [black th, pink black th, pink black, roly emo...   
2  [exo exo, mv exo, exo mv, kris exo, exo mv exo...   
3  [feminine feminine, bruhhh am not, am not, am ...   
4  [1 5, thailand s most, the world yeahhhhh, s m...   

                                 Representative_Docs  
0  [what could be so wrong with that guy that lad...  
1  [tui kh c y หัวเราะ, pink black th หัวเราะ ,, ...  
2  [who come from china or korea because that gir...  
3  [pick me girl shit 

In [ ]:
DEFAULT_PATH = "/content/drive/MyDrive/Senior-Project"
ENCODER = "bert_cv_23"

In [ ]:
# 1. Add the Topic ID back to your original DataFrame
df['topic'] = topics

# 2. Add the Top Keywords for that topic to the DataFrame (Optional, but helpful)
# This creates a dictionary map: {0: "drug_health_hospital", 1: "bank_money_stock"...}
topic_labels = topic_model.get_topic_info().set_index('Topic')['Name'].to_dict()
df['topic_name'] = df['topic'].map(topic_labels)



# 3. Save Documents + Topics to CSV
df.to_csv(f"{DEFAULT_PATH}/my_thai_topics_results_{ENCODER}.csv", index=False, encoding='utf-8-sig')

# 4. Save the Topic Definitions (The keywords list) separately
topic_info = topic_model.get_topic_info()
topic_info.to_csv(f"{DEFAULT_PATH}/my_topic_definitions_{ENCODER}.csv", index=False, encoding='utf-8-sig')

print("Files saved successfully!")

Files saved successfully!


In [ ]:
# Save the distance map
fig = topic_model.visualize_topics()
fig.write_html(f"{DEFAULT_PATH}/topic_visualization_{ENCODER}.html")

# Save the bar chart
fig_bar = topic_model.visualize_barchart()
fig_bar.write_html(f"{DEFAULT_PATH}/topic_barchart_{ENCODER}.html")

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart()

# Sweep


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 95.8 MB/s eta 0:00:00


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import pandas as pd
import gensim.corpora as corpora
from gensim.models import CoherenceModel

In [ ]:
ngram_candidates = [(1,1), (1,2), (1,3), (2,2), (2,3)]
results = []

In [ ]:
# embedding_model = SentenceTransformer("BAAI/bge-m3")

embedding_model = SentenceTransformer("airesearch/wangchanberta-base-att-spm-uncased")

embeddings = embedding_model.encode(raw_text_list, show_progress_bar=True)

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
def custom_topic_diversity(model):
    all_words = []
    topic_dict = model.get_topics()

    for topic_id, word_list in topic_dict.items():
        if word_list:
            words = [w for w, _ in word_list]
            all_words.extend(words)

    if not all_words:
        return 0.0

    return len(set(all_words)) / len(all_words)

In [ ]:
import numpy as np
import gensim.corpora as corpora
from gensim.models import CoherenceModel

def compute_gensim_coherence(model, docs_tokens):

    clean_topics = []

    for topic_id in model.get_topic_info()["Topic"].tolist():
        if topic_id == -1:
            continue

        topic_words = model.get_topic(topic_id)

        # Skip invalid
        if topic_words is None:
            continue
        if isinstance(topic_words, np.ndarray):
            topic_words = topic_words.tolist()
        if not isinstance(topic_words, list):
            continue
        if len(topic_words) == 0:
            continue

        # Extract only words
        words = []
        for w, score in topic_words:
            # skip None or non-string
            if isinstance(w, str) and w.strip() != "":
                words.append(w)

        # Skip if empty after filtering
        if len(words) == 0:
            continue

        clean_topics.append(words)

    # If no valid topics → coherence = 0
    if len(clean_topics) == 0:
        return 0.0

    # Gensim dictionary
    dictionary = corpora.Dictionary(docs_tokens)
    corpus = [dictionary.doc2bow(text) for text in docs_tokens]

    cm = CoherenceModel(
        topics=clean_topics,
        texts=docs_tokens,
        dictionary=dictionary,
        coherence="c_v",
    )

    return cm.get_coherence()

In [ ]:
def compute_metrics(model, docs_tokens):
    # Topic Diversity
    diversity = custom_topic_diversity(model)

    # Gensim Coherence (c_v)
    coherence = compute_gensim_coherence(model, docs_tokens)

    # Number of valid topics
    info = model.get_topic_info()
    n_topics = len(info[info.Topic != -1])

    return diversity, coherence, n_topics

In [ ]:
import pandas as pd
import numpy as np
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

for ngram in ngram_candidates:
    print(f"\n=== Testing ngram={ngram} ===")

    # A. Train Model
    vectorizer = CountVectorizer(tokenizer=lambda x: x.split(), token_pattern=None, ngram_range=ngram)
    model = BERTopic(vectorizer_model=vectorizer, verbose=False)
    model.fit(token_text_list, embeddings=embeddings)

    # B. Re-Tokenize Reference Text (Crucial for N-Grams)
    # This ensures "New York" exists as a single token in our dictionary
    analyzer = vectorizer.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in token_text_list]

    # C. Build Dictionary
    dictionary = corpora.Dictionary(tokenized_docs)

    # --- THE FIX: Convert Topics to IDs Manually ---
    topic_ids_list = []
    topic_info = model.get_topic_info()

    for topic_id in topic_info["Topic"]:
        if topic_id == -1: continue

        # 1. Get words from BERTopic
        raw_topic = model.get_topic(topic_id)

        # 2. Convert to String
        words = [str(item[0]) for item in raw_topic if item[0]]

        # 3. Convert to IDs (The Safe Step)
        # We only keep words that actually exist in the dictionary.
        # This prevents "KeyError" and "ValueError" inside Gensim.
        ids = [dictionary.token2id[w] for w in words if w in dictionary.token2id]

        if ids:
            topic_ids_list.append(ids)

    # D. Score it
    if not topic_ids_list:
        print("   -> ⚠️ No valid topics found matching the dictionary.")
        continue

    try:
        # Note: We pass 'topics=topic_ids_list' (Integers), not strings!
        cm = CoherenceModel(
            topics=topic_ids_list,     # Passing IDs
            texts=tokenized_docs,      # Reference text
            dictionary=dictionary,
            coherence='c_v'
        )
        score = cm.get_coherence()

        print(f"   -> Topics: {len(topic_ids_list)}, Coherence: {score:.4f}")
        results.append({"ngram": ngram, "coherence": score})

    except Exception as e:
        print(f"   -> ❌ Error: {e}")

# Print Winner
if results:
    best = max(results, key=lambda x: x['coherence'])
    print(f"\n🏆 Best N-Gram Range: {best['ngram']} with score {best['coherence']:.4f}")


=== Testing ngram=(1, 1) ===
   -> Topics: 339, Coherence: 0.4583

=== Testing ngram=(1, 2) ===
   -> Topics: 342, Coherence: 0.5703

=== Testing ngram=(1, 3) ===
   -> Topics: 349, Coherence: 0.7113

=== Testing ngram=(2, 2) ===
   -> Topics: 257, Coherence: 0.6911

=== Testing ngram=(2, 3) ===
   -> Topics: 237, Coherence: 0.8310

🏆 Best N-Gram Range: (2, 3) with score 0.8310


In [ ]:
docs_tokens = df["tokens"].tolist()
print(type(docs_tokens), type(docs_tokens[0]), docs_tokens[0][:5])

<class 'list'> <class 'list'> ['หาก', 'ต้อง', 'การปรึกษา', 'แบบ', 'ส่วนตัว']
